# 0.7 Node Tables - Refined Module Approach

**Purpose**: Build comprehensive school node tables using the refactored `NodeTableBuilder` module.

**What this notebook does**:
- Consolidates data from multiple sources into validated GeoDataFrames
- Creates public, private, and combined node tables
- Performs spatial integration (geometry, admin boundaries)
- Applies tiered validation (required/core/complete)
- Exports to multiple formats (GeoPackage, CSV, Parquet)
- Generates comprehensive quality reports

**Outputs**:
- `output/public_nodes.gpkg` - Public school node table
- `output/private_nodes.gpkg` - Private school node table
- `output/all_nodes.gpkg` - Combined node table
- `output/data_quality_report.csv` - Quality metrics report

**Next Step**: Notebook 1.0 will use these node tables for graph generation.

---
## Setup

In [1]:
# Standard imports
import sys
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
from importlib import reload
from pathlib import Path

# Project configuration
sys.path.insert(0, '../')
from config import setup_notebook
setup_notebook()

✓ Project root: /workspace/project_paaral/notebooks/..
✓ Working directory: /workspace/project_paaral
✓ Python path updated


{'project_root': PosixPath('/workspace/project_paaral/notebooks/..'),
 'data': PosixPath('/workspace/project_paaral/notebooks/../data'),
 'modules': PosixPath('/workspace/project_paaral/notebooks/../modules'),
 'notebooks': PosixPath('/workspace/project_paaral/notebooks/../notebooks'),
 'output': PosixPath('/workspace/project_paaral/notebooks/../output'),
 'psgc_shapefiles': PosixPath('/workspace/project_paaral/notebooks/../data/philippines-psgc-shapefiles/dist')}

In [2]:
# Import NodeTableBuilder
from modules import node_table_builder
reload(node_table_builder)

from modules.node_table_builder import NodeTableBuilder

# Configure display
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
sns.set_style('whitegrid')

print("Setup complete!")

Setup complete!


---
## Section 1: Build Node Tables

Initialize the builder and create comprehensive node tables with:
- Point geometries (EPSG:4326)
- Administrative boundary assignments
- Computed totals (enrollment, seats, utilization)
- Multi-tier validation

### 1.1 Initialize Builder

In [3]:
# Initialize NodeTableBuilder
builder = NodeTableBuilder(
    verbose=False,
    psgc_geodata_path='output/consolidated_geodata_matched.gpkg',  # Relative to project root
    validation_level='complete'  # Options: 'required', 'core', 'complete'
)

print("\nNodeTableBuilder initialized!")
print(f"Validation level: {builder.validation_level}")

INFO:modules.node_table_builder:NodeTableBuilder initialized (validation_level=complete)



NodeTableBuilder initialized!
Validation level: complete


### 1.2 Build Public Node Table

In [4]:
# Build public school node table
public_nodes = builder.build_public_node_table(
    include_geometry=True,      # Create Point geometry from lat/lon
    assign_boundaries=True,     # Spatial join with PSGC boundaries
    compute_totals=True         # Compute total_enrollment, total_seats, etc.
)

print("\n" + "="*60)
print("PUBLIC NODE TABLE PREVIEW")
print("="*60)
print(f"Type: {type(public_nodes)}")
print(f"Shape: {public_nodes.shape}")
print(f"CRS: {public_nodes.crs if isinstance(public_nodes, gpd.GeoDataFrame) else 'N/A'}")
print(f"\nColumns ({len(public_nodes.columns)}):")
print(list(public_nodes.columns))

INFO:modules.node_table_builder:============================================================
INFO:modules.node_table_builder:Building PUBLIC node table
INFO:modules.node_table_builder:============================================================
INFO:modules.node_table_builder:Step 1/7: Loading public coordinates...
INFO:modules.node_table_builder:  Loaded 47,821 public schools
INFO:modules.node_table_builder:Step 2/7: Loading and merging enrollment data...
INFO:modules.node_table_builder:  Merged enrollment: 47,817/47,821 schools have data
INFO:modules.node_table_builder:Step 3/7: Loading and merging facilities data...
INFO:modules.node_table_builder:  Merged facilities: 47,817/47,821 schools have data
INFO:modules.node_table_builder:Step 4/7: Loading and merging seats data...
INFO:modules.node_table_builder:  Merged seats: 45,771/47,821 schools have data
INFO:modules.node_table_builder:  Cleaning seats data based on education levels offered...
INFO:modules.node_table_builder:    Set 8


PUBLIC NODE TABLE PREVIEW
Type: <class 'geopandas.geodataframe.GeoDataFrame'>
Shape: (47821, 38)
CRS: EPSG:4326

Columns (38):
['school_id', 'school_name', 'latitude', 'longitude', 'coordinates_valid', 'enrollment_es', 'enrollment_jhs', 'enrollment_shs', 'has_enrollment_data', 'offers_es', 'offers_jhs', 'offers_shs', 'es_classrooms_instructional', 'es_classrooms_non_instructional', 'jhs_classrooms_instructional', 'jhs_classrooms_non_instructional', 'shs_classrooms_instructional', 'shs_classrooms_non_instructional', 'has_facilities_data', 'seats_es', 'seats_jhs', 'seats_shs', 'has_seats_data', 'geometry', 'adm2_pcode', 'adm1_pcode', 'region', 'province', 'adm3_psgc', 'municipality', 'admin_assignment_valid', 'total_enrollment', 'total_seats', 'capacity_utilization', 'validation_level_1', 'validation_level_2', 'validation_level_3', 'all_valid']


In [5]:
# Display sample records
display_cols = ['school_id', 'latitude', 'longitude', 'province', 'municipality', 
                'total_enrollment', 'total_seats', 'capacity_utilization', 'all_valid']
existing_display_cols = [col for col in display_cols if col in public_nodes.columns]

public_nodes[existing_display_cols].head(10)

,school_id,latitude,longitude,province,municipality,total_enrollment,total_seats,capacity_utilization,all_valid
0,100001,18.266860,120.614372,Ilocos Norte,Bacarra,49.0,195.0,0.251282,True
1,100002,18.251272,120.609487,Ilocos Norte,Bacarra,351.0,731.0,0.480164,True
2,100003,18.234670,120.616050,Ilocos Norte,Bacarra,110.0,192.0,0.572917,True
3,100004,18.250121,120.587415,Ilocos Norte,Bacarra,74.0,134.0,0.552239,True
4,100005,18.294093,120.641002,Ilocos Norte,Bacarra,45.0,44.0,1.022727,True
5,100006,18.266493,120.646128,Ilocos Norte,Bacarra,79.0,112.0,0.705357,True
6,100007,18.234491,120.608380,Ilocos Norte,Bacarra,115.0,199.0,0.577889,True
7,100008,18.270130,120.627230,Ilocos Norte,Bacarra,193.0,345.0,0.559420,True
8,100009,18.244422,120.593204,Ilocos Norte,Bacarra,125.0,90.0,1.388889,True
9,100010,18.251834,120.611409,Ilocos Norte,Bacarra,174.0,310.0,0.561290,True


In [6]:
with pd.option_context('display.max_columns', None, 'display.max_rows', 5):
    display(public_nodes)

,school_id,school_name,latitude,longitude,coordinates_valid,enrollment_es,enrollment_jhs,enrollment_shs,has_enrollment_data,offers_es,offers_jhs,offers_shs,es_classrooms_instructional,es_classrooms_non_instructional,jhs_classrooms_instructional,jhs_classrooms_non_instructional,shs_classrooms_instructional,shs_classrooms_non_instructional,has_facilities_data,seats_es,seats_jhs,seats_shs,has_seats_data,geometry,adm2_pcode,adm1_pcode,region,province,adm3_psgc,municipality,admin_assignment_valid,total_enrollment,total_seats,capacity_utilization,validation_level_1,validation_level_2,validation_level_3,all_valid
0,100001,Apaleng-Libtong ES,18.266860,120.614372,True,49.0,0.0,0.0,True,True,False,False,2.0,NaN,NaN,NaN,NaN,NaN,True,195.0,NaN,NaN,True,POINT (120.61437 18.26686),PH01028,PH01,Region I (Ilocos Region),Ilocos Norte,0102802000,Bacarra,True,49.0,195.0,0.251282,True,True,True,True
1,100002,Bacarra CES,18.251272,120.609487,True,351.0,0.0,0.0,True,True,False,False,22.0,16.0,NaN,NaN,NaN,NaN,True,731.0,NaN,NaN,True,POINT (120.60949 18.25127),PH01028,PH01,Region I (Ilocos Region),Ilocos Norte,0102802000,Bacarra,True,351.0,731.0,0.480164,True,True,True,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
47819,305414,Sagad High School,NaN,NaN,False,0.0,3023.0,0.0,True,False,True,False,NaN,NaN,0.0,0.0,NaN,NaN,True,NaN,1946.0,NaN,True,None,None,None,None,None,None,None,False,3023.0,1946.0,1.553443,False,False,False,False
47820,136725,Buting Elementary School,NaN,NaN,False,826.0,0.0,0.0,True,True,False,False,NaN,NaN,NaN,NaN,NaN,NaN,True,NaN,NaN,NaN,False,None,None,None,None,None,None,None,False,826.0,0.0,NaN,False,False,False,False


### 1.3 Build Private Node Table

In [7]:
# Build private school node table
private_nodes = builder.build_private_node_table(
    include_geometry=True,
    assign_boundaries=True,
    compute_totals=True
)

print("\n" + "="*60)
print("PRIVATE NODE TABLE PREVIEW")
print("="*60)
print(f"Type: {type(private_nodes)}")
print(f"Shape: {private_nodes.shape}")
print(f"CRS: {private_nodes.crs if isinstance(private_nodes, gpd.GeoDataFrame) else 'N/A'}")
print(f"\nColumns ({len(private_nodes.columns)}):")
print(list(private_nodes.columns))

INFO:modules.node_table_builder:============================================================
INFO:modules.node_table_builder:Building PRIVATE node table
INFO:modules.node_table_builder:============================================================
INFO:modules.node_table_builder:Step 1/7: Loading private coordinates...
INFO:modules.node_table_builder:  Loaded 11,831 private schools
INFO:modules.node_table_builder:  Parsing curricular offerings...
INFO:modules.node_table_builder:  Parsing curricular offerings from modified_coc...
INFO:modules.node_table_builder:    Parsed 11,794/11,794 schools with curricular offering data
INFO:modules.node_table_builder:    Offers ES: 9,666
INFO:modules.node_table_builder:    Offers JHS: 5,464
INFO:modules.node_table_builder:    Offers SHS: 4,757
INFO:modules.node_table_builder:Step 2/7: Loading and merging GASTPE data...
INFO:modules.node_table_builder:  Aggregated GASTPE data: 5,188 schools
INFO:modules.node_table_builder:    ESC delivering: 3,621
INFO


PRIVATE NODE TABLE PREVIEW
Type: <class 'geopandas.geodataframe.GeoDataFrame'>
Shape: (11831, 43)
CRS: EPSG:4326

Columns (43):
['school_id', 'school_name', 'latitude', 'longitude', 'coordinates_valid', 'region_left', 'modified_coc', 'offers_es', 'offers_jhs', 'offers_shs', 'esc_average_misc_fees', 'esc_average_other_fees', 'esc_average_tuition_fees', 'esc_delivering', 'shsvp_average_tuition_fees', 'shsvp_average_other_fees', 'shsvp_average_misc_fees', 'shsvp_delivering', 'has_gastpe_data', 'seats_es', 'seats_jhs', 'seats_shs', 'has_furniture_data', 'enrollment_es', 'enrollment_jhs', 'enrollment_shs', 'has_enrollment_data', 'geometry', 'adm2_pcode', 'adm1_pcode', 'region_right', 'province', 'adm3_psgc', 'municipality', 'admin_assignment_valid', 'region', 'total_enrollment', 'total_seats', 'capacity_utilization', 'validation_level_1', 'validation_level_2', 'validation_level_3', 'all_valid']


In [8]:
private_nodes.sample(3)

,school_id,school_name,latitude,longitude,coordinates_valid,region_left,modified_coc,offers_es,offers_jhs,offers_shs,esc_average_misc_fees,esc_average_other_fees,esc_average_tuition_fees,esc_delivering,shsvp_average_tuition_fees,shsvp_average_other_fees,shsvp_average_misc_fees,shsvp_delivering,has_gastpe_data,seats_es,seats_jhs,seats_shs,has_furniture_data,enrollment_es,enrollment_jhs,enrollment_shs,has_enrollment_data,geometry,adm2_pcode,adm1_pcode,region_right,province,adm3_psgc,municipality,admin_assignment_valid,region,total_enrollment,total_seats,capacity_utilization,validation_level_1,validation_level_2,validation_level_3,all_valid
4133,468527,"Bereans Alliance Learning Center, Inc.",6.5203143,124.660061,True,Region XII,Purely ES,True,False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,30.0,0.0,0.0,True,0.0,0.0,0.0,True,POINT (124.66006 6.52031),PH12063,PH12,Region XII (SOCCSKSARGEN),South Cotabato,1206311000,Norala,True,<NA>,0.0,30.0,0.000000,True,True,True,True
3450,405290,Mary-Infant Jesus School (Iligan),8.23049,124.238457,True,Region X,Purely ES,True,False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,111.0,0.0,0.0,True,114.0,0.0,0.0,True,POINT (124.23846 8.23049),PH10035,PH10,Region X (Northern Mindanao),Lanao del Norte,1003520000,Tagoloan,True,<NA>,114.0,111.0,1.027027,True,True,True,True
9372,404590,Negros Academy,9.8501,123.1389,True,Region VII,Purely JHS,False,True,False,NaN,4650.0,4650.0,True,NaN,NaN,NaN,False,True,0.0,330.0,0.0,True,0.0,629.0,0.0,True,POINT (123.1389 9.8501),PH07046,PH07,Region VII (Central Visayas),Negros Oriental,0704602000,Ayungon,True,<NA>,629.0,330.0,1.906061,True,True,True,True


In [9]:
# Display sample records
display_cols = ['school_id', 'school_name', 'latitude', 'longitude', 'province', 
                'municipality', 'modified_coc', 'total_enrollment', 'total_seats', 
                'esc_delivering', 'shsvp_delivering', 'all_valid']
existing_display_cols = [col for col in display_cols if col in private_nodes.columns]

private_nodes[existing_display_cols].head(4)

,school_id,school_name,latitude,longitude,province,municipality,modified_coc,total_enrollment,total_seats,esc_delivering,shsvp_delivering,all_valid
0,400860,Data Center College of the Philippines,17.58521,120.6149,Abra,Bangued,Purely SHS,157.0,0.0,False,True,False
1,406102,Abra Valley Colleges,17.59657,120.6145,Abra,Bangued,All Offering,270.0,772.0,True,True,True
2,406104,Divine Word College-Bangued,17.59447,120.61364,Abra,Bangued,JHS with SHS,973.0,1214.0,True,True,True
3,406107,Holy Spirit Academy of Bangued,17.596118,120.617597,Abra,Bangued,All Offering,878.0,1396.0,True,True,True


### 1.4 Build Combined Node Table

In [ ]:
# Build combined public + private node table
all_nodes = builder.build_combined_node_table(
    include_geometry=True,
    assign_boundaries=True,
    compute_totals=True
)

print("\n" + "="*60)
print("COMBINED NODE TABLE PREVIEW")
print("="*60)
print(f"Type: {type(all_nodes)}")
print(f"Shape: {all_nodes.shape}")
print(f"\nSchools by sector:")
print(all_nodes['sector'].value_counts())

In [ ]:
# Display sample records from both sectors
display_cols = ['school_id', 'sector', 'province', 'municipality', 
                'total_enrollment', 'total_seats', 'all_valid']
existing_display_cols = [col for col in display_cols if col in all_nodes.columns]

print("Sample public schools:")
display(all_nodes[all_nodes['sector'] == 'public'][existing_display_cols].head(5))

print("\nSample private schools:")
display(all_nodes[all_nodes['sector'] == 'private'][existing_display_cols].head(5))

---
## Section 2: Data Quality Review

Examine validation results, completeness statistics, and spatial coverage.

### 2.1 Summary Statistics

In [60]:
# Get comprehensive summary
summary = builder.get_summary()

# Display as formatted JSON
import json
print("\n" + "="*60)
print("COMPREHENSIVE SUMMARY")
print("="*60)
print(json.dumps(summary, indent=2))


COMPREHENSIVE SUMMARY
{
  "public": {
    "total_schools": 47821,
    "validation_breakdown": {
      "level_1_required": 47116,
      "level_2_core": 47112,
      "level_3_complete": 45315,
      "all_valid": 45315
    },
    "completeness_by_source": {
      "enrollment": {
        "count": 47817,
        "percentage": 100.0
      },
      "facilities": {
        "count": 47817,
        "percentage": 100.0
      },
      "seats": {
        "count": 45771,
        "percentage": 95.7
      }
    },
    "spatial_coverage": {
      "valid_coordinates": 47116,
      "invalid_coordinates": 705
    },
    "computed_metrics": {
      "schools_with_enrollment": 47821,
      "total_enrollment_sum": 21238937,
      "schools_with_seats": 47821,
      "total_seats_sum": 18368604,
      "schools_with_utilization": 45771,
      "avg_capacity_utilization": 1.57
    }
  },
  "private": {
    "total_schools": 11831,
    "validation_breakdown": {
      "level_1_required": 10487,
      "level_2_core": 

### 2.2 Public School Summary

In [ ]:
# Get public-only summary
public_summary = builder.get_public_summary()

print("\n" + "="*60)
print("PUBLIC SCHOOL SUMMARY")
print("="*60)
print(json.dumps(public_summary, indent=2))

### 2.3 Private School Summary

In [ ]:
# Get private-only summary
private_summary = builder.get_private_summary()

print("\n" + "="*60)
print("PRIVATE SCHOOL SUMMARY")
print("="*60)
print(json.dumps(private_summary, indent=2))

### 2.4 Validation Report

In [ ]:
# Get validation report (schools with issues)
validation_report = builder.get_validation_report()

print(f"\nSchools with validation issues: {len(validation_report):,}")
print("\nSample of validation issues:")
validation_report.head(20)

In [ ]:
# Analyze validation issues by type
if len(validation_report) > 0:
    print("\nValidation issues by type:")
    issue_counts = validation_report['validation_issues'].value_counts()
    print(issue_counts)

### 2.5 Validation Breakdown Visualization

In [ ]:
# Visualize validation levels
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Public schools
if 'validation_level_1' in public_nodes.columns:
    validation_counts = {
        'Level 1\n(Required)': public_nodes['validation_level_1'].sum(),
        'Level 2\n(Core)': public_nodes['validation_level_2'].sum(),
        'Level 3\n(Complete)': public_nodes['validation_level_3'].sum()
    }
    
    axes[0].bar(validation_counts.keys(), validation_counts.values(), color=['#ff7f0e', '#2ca02c', '#1f77b4'])
    axes[0].set_title('Public Schools - Validation Levels', fontsize=14, fontweight='bold')
    axes[0].set_ylabel('Number of Schools')
    axes[0].set_ylim(0, len(public_nodes) * 1.1)
    
    # Add value labels
    for i, (label, value) in enumerate(validation_counts.items()):
        axes[0].text(i, value + len(public_nodes)*0.02, f'{value:,}\n({100*value/len(public_nodes):.1f}%)', 
                    ha='center', va='bottom', fontweight='bold')

# Private schools
if 'validation_level_1' in private_nodes.columns:
    validation_counts = {
        'Level 1\n(Required)': private_nodes['validation_level_1'].sum(),
        'Level 2\n(Core)': private_nodes['validation_level_2'].sum(),
        'Level 3\n(Complete)': private_nodes['validation_level_3'].sum()
    }
    
    axes[1].bar(validation_counts.keys(), validation_counts.values(), color=['#ff7f0e', '#2ca02c', '#1f77b4'])
    axes[1].set_title('Private Schools - Validation Levels', fontsize=14, fontweight='bold')
    axes[1].set_ylabel('Number of Schools')
    axes[1].set_ylim(0, len(private_nodes) * 1.1)
    
    # Add value labels
    for i, (label, value) in enumerate(validation_counts.items()):
        axes[1].text(i, value + len(private_nodes)*0.02, f'{value:,}\n({100*value/len(private_nodes):.1f}%)', 
                    ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

### 2.6 Data Completeness Visualization

In [ ]:
# Visualize data source completeness
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Public schools
if public_summary and 'completeness_by_source' in public_summary:
    sources = list(public_summary['completeness_by_source'].keys())
    percentages = [public_summary['completeness_by_source'][s]['percentage'] for s in sources]
    
    axes[0].barh(sources, percentages, color='steelblue')
    axes[0].set_title('Public Schools - Data Completeness', fontsize=14, fontweight='bold')
    axes[0].set_xlabel('Completeness (%)')
    axes[0].set_xlim(0, 105)
    
    # Add percentage labels
    for i, (source, pct) in enumerate(zip(sources, percentages)):
        axes[0].text(pct + 1, i, f'{pct:.1f}%', va='center', fontweight='bold')

# Private schools
if private_summary and 'completeness_by_source' in private_summary:
    sources = list(private_summary['completeness_by_source'].keys())
    percentages = [private_summary['completeness_by_source'][s]['percentage'] for s in sources]
    
    axes[1].barh(sources, percentages, color='coral')
    axes[1].set_title('Private Schools - Data Completeness', fontsize=14, fontweight='bold')
    axes[1].set_xlabel('Completeness (%)')
    axes[1].set_xlim(0, 105)
    
    # Add percentage labels
    for i, (source, pct) in enumerate(zip(sources, percentages)):
        axes[1].text(pct + 1, i, f'{pct:.1f}%', va='center', fontweight='bold')

plt.tight_layout()
plt.show()

### 2.7 Spatial Coverage Map

In [ ]:
# Plot spatial distribution of schools
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# Public schools
if isinstance(public_nodes, gpd.GeoDataFrame):
    valid_public = public_nodes[public_nodes['coordinates_valid'] == True]
    invalid_public = public_nodes[public_nodes['coordinates_valid'] == False]
    
    valid_public.plot(ax=axes[0], markersize=1, color='green', alpha=0.5, label='Valid')
    if len(invalid_public) > 0:
        invalid_public.plot(ax=axes[0], markersize=3, color='red', alpha=0.8, label='Invalid')
    
    axes[0].set_title(f'Public Schools (n={len(public_nodes):,})', fontsize=14, fontweight='bold')
    axes[0].set_xlabel('Longitude')
    axes[0].set_ylabel('Latitude')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

# Private schools
if isinstance(private_nodes, gpd.GeoDataFrame):
    valid_private = private_nodes[private_nodes['coordinates_valid'] == True]
    invalid_private = private_nodes[private_nodes['coordinates_valid'] == False]
    
    valid_private.plot(ax=axes[1], markersize=1, color='blue', alpha=0.5, label='Valid')
    if len(invalid_private) > 0:
        invalid_private.plot(ax=axes[1], markersize=3, color='red', alpha=0.8, label='Invalid')
    
    axes[1].set_title(f'Private Schools (n={len(private_nodes):,})', fontsize=14, fontweight='bold')
    axes[1].set_xlabel('Longitude')
    axes[1].set_ylabel('Latitude')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 2.8 Capacity Utilization Analysis

In [ ]:
# Analyze capacity utilization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Public schools
if 'capacity_utilization' in public_nodes.columns:
    valid_util = public_nodes['capacity_utilization'].dropna()
    # Filter to reasonable range for visualization (0-3)
    valid_util_filtered = valid_util[(valid_util >= 0) & (valid_util <= 3)]
    
    axes[0].hist(valid_util_filtered, bins=50, color='steelblue', edgecolor='black', alpha=0.7)
    axes[0].axvline(1.0, color='red', linestyle='--', linewidth=2, label='100% capacity')
    axes[0].set_title(f'Public Schools - Capacity Utilization (n={len(valid_util_filtered):,})', 
                     fontsize=14, fontweight='bold')
    axes[0].set_xlabel('Capacity Utilization (enrollment/seats)')
    axes[0].set_ylabel('Number of Schools')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

# Private schools
if 'capacity_utilization' in private_nodes.columns:
    valid_util = private_nodes['capacity_utilization'].dropna()
    # Filter to reasonable range for visualization (0-3)
    valid_util_filtered = valid_util[(valid_util >= 0) & (valid_util <= 3)]
    
    axes[1].hist(valid_util_filtered, bins=50, color='coral', edgecolor='black', alpha=0.7)
    axes[1].axvline(1.0, color='red', linestyle='--', linewidth=2, label='100% capacity')
    axes[1].set_title(f'Private Schools - Capacity Utilization (n={len(valid_util_filtered):,})', 
                     fontsize=14, fontweight='bold')
    axes[1].set_xlabel('Capacity Utilization (enrollment/seats)')
    axes[1].set_ylabel('Number of Schools')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## Section 3: Export Node Tables

Export to multiple formats for downstream analysis.

### 3.1 Export to GeoPackage (Primary Format)

In [12]:
# Export public schools to GeoPackage
builder.export_geopackage(
    path='output/public_nodes.gpkg',
    sector='public',
    valid_only=False  # Export all schools (including invalid for inspection)
)

INFO:pyogrio._io:Created 47,821 records
INFO:modules.node_table_builder:Exported 47,821 schools to /workspace/project_paaral/output/public_nodes.gpkg


In [13]:
# Export private schools to GeoPackage
builder.export_geopackage(
    path='output/private_nodes.gpkg',
    sector='private',
    valid_only=False
)

INFO:pyogrio._io:Created 11,831 records
INFO:modules.node_table_builder:Exported 11,831 schools to /workspace/project_paaral/output/private_nodes.gpkg


In [ ]:
# # Export combined table to GeoPackage (for complete network analysis)
# builder.export_geopackage(
#     path='output/all_nodes.gpkg',
#     sector='both',
#     valid_only=False
# )

### 3.2 Export Valid Schools Only (for Graph Generation)

In [10]:
# Export only valid schools for graph generation (Notebook 1.0)
builder.export_geopackage(
    path='output/private_nodes_valid.gpkg',
    sector='private',
    valid_only=True  # Only schools passing validation
)

# print("\nValid schools exported for graph generation!")

INFO:modules.node_table_builder:Filtered to 9,305 valid schools
INFO:pyogrio._io:Created 9,305 records
INFO:modules.node_table_builder:Exported 9,305 schools to /workspace/project_paaral/output/private_nodes_valid.gpkg


In [11]:
# Export only valid schools for graph generation (Notebook 1.0)
builder.export_geopackage(
    path='output/public_nodes_valid.gpkg',
    sector='public',
    valid_only=True  # Only schools passing validation
)

# print("\nValid schools exported for graph generation!")

INFO:modules.node_table_builder:Filtered to 44,899 valid schools
INFO:pyogrio._io:Created 44,899 records
INFO:modules.node_table_builder:Exported 44,899 schools to /workspace/project_paaral/output/public_nodes_valid.gpkg


### 3.3 Export to CSV (for non-spatial analysis)

In [ ]:
# # Export to CSV (geometry column dropped)
# builder.export_csv(
#     path='output/public_nodes.csv',
#     sector='public',
#     valid_only=False
# )

# builder.export_csv(
#     path='output/private_nodes.csv',
#     sector='private',
#     valid_only=False
# )

# builder.export_csv(
#     path='output/all_nodes.csv',
#     sector='both',
#     valid_only=False
# )

### 3.4 Export to Parquet (memory-efficient format)

In [ ]:
# Export to Parquet (supports GeoDataFrame, more efficient than CSV)
builder.export_parquet(
    path='output/node_tables/public_node_table.parquet',
    sector='public',
    valid_only=False
)

print("Exported to Parquet format!")

In [ ]:
# Export to Parquet (supports GeoDataFrame, more efficient than CSV)
builder.export_parquet(
    path='output/node_tables/private_node_table.parquet',
    sector='private',
    valid_only=False
)

print("Exported to Parquet format!")

### 3.5 Export Quality Report

In [ ]:
# Export comprehensive quality report
builder.export_quality_report('output/data_quality_report.csv')

# Read and display report
quality_report = pd.read_csv('output/data_quality_report.csv')
print("\nQuality Report:")
quality_report

---
## Section 4: Provincial Filtering Example

Demonstrate filtering by province for provincial graph generation.

In [ ]:
# Example: Filter schools in Bulacan province (matches road network PH03014_bulacan.geojsonl)
bulacan_schools = all_nodes[all_nodes['adm2_pcode'] == 'PH03014'].copy()

print(f"\nSchools in Bulacan: {len(bulacan_schools):,}")
print(f"  Public: {len(bulacan_schools[bulacan_schools['sector'] == 'public']):,}")
print(f"  Private: {len(bulacan_schools[bulacan_schools['sector'] == 'private']):,}")
print(f"  Valid: {bulacan_schools['all_valid'].sum():,}")

# Display sample
display_cols = ['school_id', 'sector', 'municipality', 'total_enrollment', 'total_seats', 'all_valid']
existing_display_cols = [col for col in display_cols if col in bulacan_schools.columns]
bulacan_schools[existing_display_cols].head(10)

In [ ]:
# Visualize Bulacan schools
if isinstance(bulacan_schools, gpd.GeoDataFrame) and len(bulacan_schools) > 0:
    fig, ax = plt.subplots(figsize=(10, 10))
    
    # Plot by sector
    bulacan_public = bulacan_schools[bulacan_schools['sector'] == 'public']
    bulacan_private = bulacan_schools[bulacan_schools['sector'] == 'private']
    
    bulacan_public.plot(ax=ax, markersize=10, color='green', alpha=0.6, label='Public')
    bulacan_private.plot(ax=ax, markersize=10, color='blue', alpha=0.6, label='Private')
    
    ax.set_title(f'Schools in Bulacan Province (n={len(bulacan_schools):,})', 
                 fontsize=14, fontweight='bold')
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

---
## Section 5: Quick Reference - Key Columns

**Identifiers & Location:**
- `school_id` - Unique school identifier
- `latitude`, `longitude` - Coordinates
- `geometry` - Shapely Point (EPSG:4326)

**Administrative Boundaries:**
- `adm1_psgc`, `adm2_psgc`, `adm3_psgc` - PSGC codes
- `adm2_pcode` - Province code (matches road network files)
- `region`, `province`, `municipality` - Human-readable names

**Enrollment & Capacity:**
- `enrollment_es`, `enrollment_jhs`, `enrollment_shs` - By level
- `seats_es`, `seats_jhs`, `seats_shs` - By level
- `total_enrollment`, `total_seats` - Aggregated totals
- `capacity_utilization` - Enrollment / seats ratio

**Validation:**
- `coordinates_valid` - Coordinates within PH bounds
- `admin_assignment_valid` - Spatially matched to province
- `validation_level_1` - Required fields present
- `validation_level_2` - Core data present
- `validation_level_3` - Complete data present
- `all_valid` - Passes selected validation level

**Data Source Flags:**
- `has_enrollment_data`, `has_seats_data`, `has_facilities_data`
- `has_gastpe_data` (private only)

**Private Schools Only:**
- `modified_coc` - Curricular offering category
- `esc_delivering`, `shsvp_delivering` - GASTPE program participation

---
## Summary

**Node tables created:**
- ✅ Public schools with complete attributes
- ✅ Private schools with complete attributes
- ✅ Combined public + private table

**Key features:**
- ✅ Point geometries (EPSG:4326)
- ✅ Administrative boundary assignments
- ✅ Provincial codes (`adm2_pcode`) for road network matching
- ✅ Computed totals for graph node weights
- ✅ Multi-tier validation

**Outputs ready for Notebook 1.0:**
- `output/all_nodes_valid.gpkg` - Valid schools for graph generation
- `output/all_nodes.gpkg` - All schools (including invalid for review)
- `output/data_quality_report.csv` - Quality metrics

**Next step**: Use these node tables in Notebook 1.0 for graph generation!